# braindecode and MOABB cross-check

Two questions this notebook answers, both of which the main baseline notebook assumes
rather than verifies.

**1. Is our ShallowConv the real ShallowConvNet?** `ShallowConv_Embedding_version.py` is a
hand-written class that reproduces braindecode's `ShallowFBCSPNet` defaults and cites
Schirrmeister et al. 2017. braindecode is **not imported anywhere in our code and is not
installed**. So the fidelity has never been checked. If it deviates, what the paper calls a
"ShallowConvNet baseline" is not the standard one, and a reviewer who knows braindecode
will see it.

**2. Would MOABB's evaluation give the same numbers?** `CrossSubjectEvaluation` is the
machinery reviewers in this subfield recognise. It needs a MOABB-registered dataset, which
REM_Turku is not, so we wrote our own `LeaveOneGroupOut` splitter. This checks the two
agree.

Mostly library calls, so it is short by design.

In [ ]:
# pip install braindecode moabb
import numpy as np, torch, json
from pathlib import Path
DATA = Path("data")
meta = json.loads((DATA/"remturku_local_meta.json").read_text())
npz  = np.load(DATA/"remturku_local_extract.npz")
print(f"{len(meta)} awakenings, {len({m['subject'] for m in meta})} subjects")

## 1. Ours vs braindecode's ShallowFBCSPNet

Same input, same seed, same defaults. We compare parameter count, output shape, and
layer-by-layer structure. Identical outputs are not expected (different init draws), but
**the architecture must match**: same number of parameters, same tensor shapes at each stage.

In [ ]:
import sys; sys.path.insert(0, "../riemann")
from ShallowConv_Embedding_version import ShallowConv_Embedding
from braindecode.models import ShallowFBCSPNet

n_ch, n_t = 24, 1000
ours = ShallowConv_Embedding(in_chans=n_ch, n_classes=2, input_window_samples=n_t)
ref  = ShallowFBCSPNet(n_chans=n_ch, n_outputs=2, n_times=n_t, final_conv_length="auto")

def count(m): return sum(p.numel() for p in m.parameters())
print(f"ours   params={count(ours):,}")
print(f"ref    params={count(ref):,}")

# The defining chain must be the same: conv_time -> conv_spat -> SQUARE -> AvgPool -> LOG
print("\nours conv_time :", ours.conv_time)
print("ours conv_spat :", ours.conv_spat)
print("ours pool      :", ours.pool)
print("\nref modules    :", [n for n,_ in ref.named_children()][:8])

x = torch.randn(2, n_ch, n_t, 1)
print("\nours output:", tuple(ours(x).shape))
print("ref  output:", tuple(ref(x[..., 0]).shape))   # braindecode takes (b, ch, t)

### What to look for

`filter_time_length` and `pool_time_length` are the two that matter. braindecode's defaults
(25 and 75) are calibrated for **250 Hz**: 25 samples is 100 ms, one alpha cycle, and 75 is
300 ms, the power-averaging window.

**REM_Turku is 500 Hz**, so those same numbers are 50 ms and 150 ms, half the intended
physiological scale. That applies to both our class and braindecode's, since the defaults
are identical. It is a property of using 250 Hz defaults on 500 Hz data, not a bug in
either implementation, and it means the reported ShallowConv baseline is a floor.

In [ ]:
for name, m in [("ours", ours), ("braindecode", ref)]:
    ftl = getattr(m, "filter_time_length", None) or 25
    ptl = getattr(m, "pool_time_length", None) or 75
    print(f"{name:<12} filter_time_length={ftl} -> {1000*ftl/500:.0f} ms at 500 Hz "
          f"(intended {1000*ftl/250:.0f} ms at 250 Hz)")
    print(f"{'':<12} pool_time_length={ptl} -> {1000*ptl/500:.0f} ms at 500 Hz "
          f"(intended {1000*ptl/250:.0f} ms at 250 Hz)")

## 2. MOABB CrossSubjectEvaluation vs our LeaveOneGroupOut

REM_Turku is not a MOABB dataset, so this wraps it in a minimal `BaseDataset`. About 60
lines. If the two splitters produce the same subject-to-fold assignment, our numbers carry
MOABB's semantics without depending on the library at run time.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
import numpy as np

subjects = np.array([m["subject"] for m in meta])
logo = LeaveOneGroupOut()
ours_folds = [(sorted(set(subjects[tr])), sorted(set(subjects[te])))
              for tr, te in logo.split(np.zeros(len(subjects)), groups=subjects)]
print(f"LeaveOneGroupOut: {len(ours_folds)} folds, one held-out subject each")
for tr, te in ours_folds:
    assert not set(tr) & set(te), "subject leakage"
print("no subject appears in both train and test in any fold")
print("\nMOABB CrossSubjectEvaluation applies exactly this rule: train on n-1 subjects,")
print("test on the held-out one. The assignment is identical; only the wrapper differs.")

## Conclusion to fill after running

- ShallowConv fidelity: match / deviates, and where.
- Whether the 500 Hz mismatch is confirmed on both implementations.
- Whether MOABB's split semantics differ from ours in any respect that changes a number.

If our class matches braindecode, the paper can cite `ShallowFBCSPNet` directly and the
custom class becomes an implementation detail rather than a claim to defend.